In [0]:
bronze_df = spark.table("paysim_fraud.bronze.transactions_raw")

display(bronze_df.limit(10))

In [0]:
silver_df = (
    bronze_df
    .withColumnRenamed("nameOrig", "origin_account")
    .withColumnRenamed("oldbalanceOrg", "origin_balance_before")
    .withColumnRenamed("newbalanceOrig", "origin_balance_after")
    .withColumnRenamed("nameDest", "destination_account")
    .withColumnRenamed("oldbalanceDest", "destination_balance_before")
    .withColumnRenamed("newbalanceDest", "destination_balance_after")
    .withColumnRenamed("isFraud", "is_fraud")
    .withColumnRenamed("isFlaggedFraud", "is_flagged_fraud")
    .withColumnRenamed("type", "transaction_type")
)

In [0]:
from pyspark.sql.functions import col, floor

silver_df = (
    silver_df
    .withColumn(
        "simulation_day",
        floor((col("step") - 1) / 24) + 1
    )
    .withColumn(
        "hour_of_day",
        (col("step") - 1) % 24
    )
)

In [0]:
from pyspark.sql.functions import when

silver_df = (
    silver_df
    .withColumn(
        "destination_type",
        when(col("destination_account").startswith("M"), "MERCHANT")
        .when(col("destination_account").startswith("C"), "CUSTOMER")
        .otherwise("UNKNOWN")
    )
)

In [0]:
silver_df = (
    silver_df
    .withColumn(
        "origin_balance_change",
        col("origin_balance_before") - col("origin_balance_after")
    )
    .withColumn(
        "destination_balance_change",
        col("destination_balance_after") - col("destination_balance_before")
    )
)

In [0]:
silver_df = (
    silver_df
    .withColumn(
        "origin_account_emptied",
        col("origin_balance_after") == 0
    )
)

In [0]:
silver_df = (
    silver_df
    .withColumn(
        "amount_to_origin_balance_ratio",
        when(
            col("origin_balance_before") > 0,
            col("amount") / col("origin_balance_before")
        )
    )
)

In [0]:
silver_df = (
    silver_df
    .withColumn("is_fraud", col("is_fraud").cast("boolean"))
    .withColumn("is_flagged_fraud", col("is_flagged_fraud").cast("boolean"))
)

In [0]:
silver_df.printSchema()

In [0]:
display(silver_df.limit(20))

In [0]:
silver_df.count()

In [0]:
display(
    silver_df
    .groupBy("transaction_type")
    .count()
    .orderBy("count", ascending=False)
)

In [0]:
display(
    silver_df
    .groupBy("is_fraud")
    .count()
)

In [0]:
(
    silver_df.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("paysim_fraud.silver.transactions")
)

In [0]:
spark.table("paysim_fraud.silver.transactions").count()